<a href="https://colab.research.google.com/github/shreyansh8h/hello-world/blob/master/masterclass_Chunking_strategies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install FAISS (Facebook AI Similarity Search) CPU version.
# FAISS is used to efficiently store, index, and search high-dimensional vectors.
# Commonly used in RAG applications to retrieve the most relevant document chunks
# based on embedding similarity before passing them to an LLM.
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 34.3 MB/s eta 0:00:00


In [ ]:
# Install spaCy, an advanced NLP library used for text processing tasks such as
# tokenization, sentence segmentation, part-of-speech tagging, named entity recognition,
# and dependency parsing. It is commonly used in text analytics, information extraction,
# and RAG pipelines for preprocessing documents.
!pip install spacy

In [ ]:
# Upgrade Gradio and Hugging Face Hub to compatible versions.
# This resolves dependency and import issues caused by older Gradio versions
# while ensuring compatibility with recent Hugging Face libraries and APIs.
# Required for building and deploying modern Gradio-based web applications.

!pip install -U gradio==5.38.2 huggingface_hub==1.1.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.9/514.9 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.5/324.5 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.7 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.4
    Uninstalling pydantic_core-2.41.4:
      Successfully uninstalled pydantic_core-2.41.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.3
    Uninstalling pydantic-2.12.3:
      Successfully uninstalled pydantic-2.12.3
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.17.0
    Uninstalling huggingface_hub-1.17.0:
      Successfully uninstalled hugging

In [ ]:
import gradio as gr           # Gradio: Create interactive web-based applications and dashboards
import re                     # re: Perform pattern matching and text processing using regular expressions
import html                   # html: Safely escape HTML special characters in text output
from typing import List, Dict # List, Dict: Add type hints for better code readability
import nltk                   # NLTK: Natural Language Toolkit for basic NLP operations
from nltk.tokenize import word_tokenize  # word_tokenize: Split text into individual words/tokens
import spacy                  # spaCy: Advanced NLP library for tokenization and sentence parsing
from dataclasses import dataclass  # dataclass: Create simple classes for storing structured data

In [ ]:
nltk.download('punkt')      # Download Punkt tokenizer models for sentence and word tokenization
nltk.download('punkt_tab')  # Download additional Punkt tokenizer data required by newer NLTK versions

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# Load the spaCy language model that will be used throughout the application
# for NLP tasks such as tokenization, sentence detection, and linguistic analysis.

try:
    # Attempt to load the pre-trained English language model.
    # 'en_core_web_sm' is a lightweight English model provided by spaCy.
    # It contains vocabulary, tokenization rules, POS tagging,
    # dependency parsing, and named entity recognition capabilities.
    nlp = spacy.load("en_core_web_sm")

except:
    # If the model is not found on the system, execution enters this block.

    # Import the subprocess module to execute terminal/command-line commands
    # directly from within the Python script.
    import subprocess

    # Run the spaCy download command from Python.
    # This downloads and installs the 'en_core_web_sm' model package.
    # Equivalent terminal command:
    # python -m spacy download en_core_web_sm
    subprocess.run(
        ["python", "-m", "spacy", "download", "en_core_web_sm"]
    )

    # After successful installation, load the model into memory
    # so it can be used for NLP processing in the application.
    nlp = spacy.load("en_core_web_sm")

In [ ]:
# Define a lightweight data container using Python's @dataclass decorator.
# This class is used to store and organize the results produced by different
# chunking methods in a structured format. It keeps the chunking method name,
# generated chunks, execution metadata, and sample tokenized outputs together,
# making it easier to pass, access, and manage chunking results throughout the application.
@dataclass
class ChunkResult:
    """Data class to capture processing metrics and text payloads uniformly"""
    method: str
    chunks: List[str]
    metadata: Dict = None
    tokens_examples: List[List[str]] = None

# Define a class that contains multiple text chunking techniques.
# This class provides different strategies for splitting large documents
# into smaller chunks based on words, sentences, paragraphs, tokens,
# or document structure. These chunking methods are commonly used in
# NLP applications, semantic search systems, and RAG (Retrieval-Augmented Generation) pipelines.
class TextChunker:
    """Implementation of standard token-aware and structural splitting strategies"""

    def __init__(self):
        self.nlp = nlp

    def fixed_size_chunking(self, text: str, chunk_size: int = 100, overlap: int = 0) -> ChunkResult:
        words = text.split()
        chunks, tokens_examples = [], []

        if overlap >= chunk_size:
            overlap = chunk_size // 4

        step = max(1, chunk_size - overlap)
        for i in range(0, len(words), step):
            chunk_words = words[i:i + chunk_size]
            if chunk_words:
                chunk_text = ' '.join(chunk_words)
                chunks.append(chunk_text)
                if len(tokens_examples) < 3:
                    tokens_examples.append(word_tokenize(chunk_text))

        metadata = {
            "total_words": len(words),
            "chunk_size_words": chunk_size,
            "overlap_words": overlap,
            "num_chunks": len(chunks)
        }
        return ChunkResult("Fixed-Size Chunking", chunks, metadata, tokens_examples)


# Sentence-Based Chunking divides a document into chunks containing
# a fixed number of complete sentences. Unlike fixed-size chunking,
# this approach preserves sentence boundaries and maintains better
# semantic meaning, making it suitable for NLP tasks, information
# retrieval, and Retrieval-Augmented Generation (RAG) systems.
    def sentence_based_chunking(self, text: str, sentences_per_chunk: int = 10) -> ChunkResult:
        doc = self.nlp(text)
        sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]
        chunks, tokens_examples = [], []

        for i in range(0, len(sentences), sentences_per_chunk):
            chunk_sentences = sentences[i:i + sentences_per_chunk]
            if chunk_sentences:
                chunk_text = ' '.join(chunk_sentences)
                chunks.append(chunk_text)
                if len(tokens_examples) < 3:
                    tokens_examples.append([t.text for t in self.nlp(chunk_text)])

        metadata = {
            "total_sentences": len(sentences),
            "sentences_per_chunk": sentences_per_chunk,
            "num_chunks": len(chunks)
        }
        return ChunkResult("Sentence-Based Chunking", chunks, metadata, tokens_examples)

# Sliding Window Chunking divides text into fixed-size chunks while maintaining
# an overlap between consecutive chunks. The overlapping words help preserve
# contextual information across chunk boundaries, reducing information loss
# and improving retrieval quality in NLP, semantic search, and RAG systems.

    def sliding_window_chunking(self, text: str, chunk_size: int = 100, overlap: int = 20) -> ChunkResult:
        words = text.split()
        chunks, tokens_examples = [], []

        if overlap >= chunk_size:
            overlap = chunk_size // 5

        step = max(1, chunk_size - overlap)
        for i in range(0, len(words), step):
            chunk_words = words[i:i + chunk_size]
            if chunk_words:
                chunk_text = ' '.join(chunk_words)
                chunks.append(chunk_text)
                if len(tokens_examples) < 3:
                    tokens_examples.append(word_tokenize(chunk_text))

        metadata = {
            "total_words": len(words),
            "chunk_size_words": chunk_size,
            "overlap_words": overlap,
            "num_chunks": len(chunks)
        }
        return ChunkResult("Sliding Window Chunking", chunks, metadata, tokens_examples)

# Paragraph-Level Chunking treats each paragraph as an independent chunk.
# This approach preserves the natural document structure by keeping related
# sentences together within the same paragraph. It is useful for documents
# such as articles, reports, research papers, and manuals where paragraphs
# represent distinct ideas or topics.
    def paragraph_level_chunking(self, text: str) -> ChunkResult:
        paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
        if len(paragraphs) <= 1:
            paragraphs = [p.strip() for p in text.split('\n') if p.strip()]

        chunks = paragraphs
        tokens_examples = [[t.text for t in self.nlp(chunk)] for chunk in chunks[:3]]

        metadata = {
            "total_paragraphs": len(paragraphs),
            "num_chunks": len(chunks)
        }
        return ChunkResult("Paragraph-Level Chunking", chunks, metadata, tokens_examples)


# Hierarchical Chunking follows a top-down document structure approach.
# It first attempts to split the document into sections, then paragraphs,
# and finally sentences if required. This strategy preserves the natural
# hierarchy of the document and only breaks content into smaller units when
# the maximum chunk size constraint is exceeded. It is particularly useful
# for structured documents such as reports, research papers, manuals,
# technical documentation, and legal documents.
    def hierarchical_chunking(self, text: str, max_chunk_size: int = 200) -> ChunkResult:
        """
        True Hierarchical Chunking: Top-Down Structural Parsing.
        Preserves complete sections or nested paragraphs unless they break max_chunk_size constraints.
        """
        chunks = []
        tokens_examples = []

        section_pattern = r'\n(?=[A-Z][A-Z\s]+\n|\d+\.\s|\n#)|(?=",\s*"[^"]+":\s*""")|(?=",\s*"[^"]+":\s*")'
        sections = re.split(section_pattern, text)

        for section in sections:
            section = section.strip().lstrip(',').strip()
            if not section:
                continue

            if len(section.split()) <= max_chunk_size:
                chunks.append(section)
                continue

            paragraphs = [p.strip() for p in section.split('\n\n') if p.strip()]
            current_para_group = []
            current_para_words = 0

            for paragraph in paragraphs:
                para_word_count = len(paragraph.split())

                if para_word_count > max_chunk_size:
                    if current_para_group:
                        chunks.append('\n\n'.join(current_para_group))
                        current_para_group = []
                        current_para_words = 0

                    doc = self.nlp(paragraph)
                    sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]
                    current_sent_group = []
                    current_sent_words = 0

                    for sent in sentences:
                        sent_word_count = len(sent.split())
                        if current_sent_words + sent_word_count > max_chunk_size:
                            if current_sent_group:
                                chunks.append(' '.join(current_sent_group))
                            current_sent_group = [sent]
                            current_sent_words = sent_word_count
                        else:
                            current_sent_group.append(sent)
                            current_sent_words += sent_word_count

                    if current_sent_group:
                        chunks.append(' '.join(current_sent_group))

                elif current_para_words + para_word_count > max_chunk_size:
                    if current_para_group:
                        chunks.append('\n\n'.join(current_para_group))
                    current_para_group = [paragraph]
                    current_para_words = para_word_count
                else:
                    current_para_group.append(paragraph)
                    current_para_words += para_word_count

            if current_para_group:
                chunks.append('\n\n'.join(current_para_group))

        for chunk in chunks[:3]:
            tokens_examples.append([t.text for t in self.nlp(chunk)])

        metadata = {
            "max_chunk_size": max_chunk_size,
            "hierarchy_tiers_processed": "Sections -> Paragraphs -> Sentences",
            "num_chunks": len(chunks)
        }
        return ChunkResult("Hierarchical Chunking", chunks, metadata, tokens_examples)

# Recursive Text Splitter uses a hierarchical splitting strategy to ensure
# that generated chunks remain within the specified size limit. Instead of
# immediately splitting text into fixed-size segments, it recursively breaks
# the document at natural boundaries such as paragraphs and sentences.
# This approach helps preserve semantic meaning while maintaining chunk size
# constraints, making it highly effective for RAG systems, semantic search,
# and document processing pipelines.
    def recursive_text_splitter(self, text: str, chunk_size: int = 200, overlap: int = 50) -> ChunkResult:
        def recursive_split(text_part, chunk_size, overlap):
            words = text_part.split()
            if len(words) <= chunk_size:
                return [text_part]

            paragraphs = text_part.split('\n\n')
            if len(paragraphs) > 1:
                chunks = []
                for para in paragraphs:
                    chunks.extend(recursive_split(para, chunk_size, overlap))
                return chunks

            doc = self.nlp(text_part)
            sentences = [sent.text.strip() for sent in doc.sents]
            chunks, current_chunk = [], []
            current_words = 0

            for sentence in sentences:
                sentence_words = len(sentence.split())
                if current_words + sentence_words > chunk_size and current_chunk:
                    chunks.append(' '.join(current_chunk))
                    overlap_sentences, overlap_words = [], 0
                    for sent in reversed(current_chunk):
                        sent_words = len(sent.split())
                        if overlap_words + sent_words <= overlap:
                            overlap_sentences.insert(0, sent)
                            overlap_words += sent_words
                        else:
                            break
                    current_chunk = overlap_sentences + [sentence]
                    current_words = overlap_words + sentence_words
                else:
                    current_chunk.append(sentence)
                    current_words += sentence_words

            if current_chunk:
                chunks.append(' '.join(current_chunk))
            return chunks

        chunks = recursive_split(text, chunk_size, overlap)
        tokens_examples = [[t.text for t in self.nlp(c)] for c in chunks[:3]]

        metadata = {
            "chunk_size_words": chunk_size,
            "overlap_words": overlap,
            "num_chunks": len(chunks)
        }
        return ChunkResult("Recursive Text Splitter", chunks, metadata, tokens_examples)


# Character Text Splitter divides text purely based on character count
# rather than words, sentences, or document structure. Each chunk contains
# a fixed number of characters, with optional overlap between consecutive
# chunks to preserve context. This method is simple, fast, and useful when
# exact character limits must be maintained, such as preparing text for
# APIs, language models, or systems with character-based constraints.
    def character_text_splitter(self, text: str, chunk_size: int = 100, overlap: int = 10) -> ChunkResult:
        chunks, tokens_examples = [], []
        if overlap >= chunk_size:
            overlap = chunk_size // 10

        step = max(1, chunk_size - overlap)
        for i in range(0, len(text), step):
            chunk = text[i:i + chunk_size]
            if chunk:
                chunks.append(chunk)
                if len(tokens_examples) < 3:
                    tokens_examples.append([t.text for t in self.nlp(chunk)])

        metadata = {
            "total_characters": len(text),
            "chunk_size_chars": chunk_size,
            "overlap_chars": overlap,
            "num_chunks": len(chunks)
        }
        return ChunkResult("Character Text Splitter", chunks, metadata, tokens_examples)


# Token Text Splitter divides text based on the number of NLP tokens
# rather than characters, words, or sentences. Tokens are generated using
# the spaCy tokenizer and may represent words, punctuation marks, numbers,
# or special symbols. This method is particularly useful for Large Language
# Models (LLMs) because model input limits are usually measured in tokens
# rather than characters or words. Optional token overlap helps preserve
# context between consecutive chunks and improves retrieval performance in
# RAG and semantic search applications.
    def token_text_splitter(self, text: str, max_tokens: int = 100, overlap_tokens: int = 20) -> ChunkResult:
        tokens = [t.text for t in self.nlp(text)]
        chunks_tokens = []

        if overlap_tokens >= max_tokens:
            overlap_tokens = max_tokens // 5

        step = max(1, max_tokens - overlap_tokens)
        for i in range(0, len(tokens), step):
            chunk_tokens = tokens[i:i + max_tokens]
            if chunk_tokens:
                chunks_tokens.append(chunk_tokens)

        chunks = [' '.join(chunk) for chunk in chunks_tokens]
        tokens_examples = chunks_tokens[:3]

        metadata = {
            "total_tokens": len(tokens),
            "max_tokens_per_chunk": max_tokens,
            "overlap_tokens": overlap_tokens,
            "num_chunks": len(chunks)
        }
        return ChunkResult("Token Text Splitter", chunks, metadata, tokens_examples)







Gradio Application Interface Configuration

In [ ]:
# Creates and configures the complete Gradio application interface.
# This function initializes the TextChunker engine, defines sample datasets,
# provides descriptions for each chunking technique, implements the text
# analysis workflow, dynamically updates UI components based on the selected
# chunking method, and constructs the interactive dashboard using Gradio.
# The resulting interface allows users to experiment with multiple chunking
# strategies, inspect generated chunks, view tokenization results, analyze
# execution statistics, and compare the behavior of different text splitting
# algorithms in real time.

def create_demo():
    chunker = TextChunker()

    sample_texts = {
        "Research Paper (AI in Healthcare)": "ARTIFICIAL INTELLIGENCE IN HEALTHCARE\n\nINTRODUCTION\n\nArtificial Intelligence (AI) is revolutionizing the healthcare industry by enabling faster, more accurate diagnoses and personalized treatment plans.\n\nAI technologies, including machine learning and deep learning algorithms, are being deployed across various medical domains. These systems can analyze complex medical data, identify patterns, and provide insights.\n\nAPPLICATIONS IN DIAGNOSIS\n\nMedical Imaging Analysis: AI algorithms can analyze medical images such as X-rays, MRIs, and CT scans with high accuracy.\n\nPredictive Analytics: Machine learning models can predict disease progression and patient outcomes based on historical data.",
        "News Article": "Technology Giant Announces Breakthrough in Quantum Computing\n\nIn a major scientific advancement, researchers have demonstrated a practical quantum computer capable of solving complex problems that were previously considered impossible.\n\nThe new system uses 256 qubits to perform calculations that would take conventional supercomputers thousands of years to complete.",
        "Short Example": "Artificial Intelligence (AI) is transforming healthcare by enabling faster diagnosis, personalized treatment, and predictive analytics."
    }

    descriptions = {
        "Fixed-Size Chunking": "## Fixed-Size Chunking\n\n**How it works**: Splits text into chunks of exactly N words each, cutting at word boundaries.\n\n**Best for**:\n- Simple document splitting\n- Uniform processing pipelines\n- When sentence structure is not important\n\n**Example**: 1000-word document → 10 chunks of 100 words each",
        "Sentence-Based Chunking": "## Sentence-Based Chunking\n\n**How it works**: Groups N sentences together to form each chunk, preserving sentence boundaries.\n\n**Best for**:\n- Natural language understanding\n- Maintaining grammatical structure\n- When sentences need to remain intact\n\n**Example**: Every 10 sentences form one chunk",
        "Sliding Window Chunking": "## Sliding Window Chunking\n\n**How it works**: Creates overlapping chunks with specified window size and overlap.\n\n**Best for**:\n- Context preservation\n- NLP models requiring context overlap\n- Preventing information loss at chunk boundaries\n\n**Example**: 100-word chunks with 20-word overlap",
        "Paragraph-Level Chunking": "## Paragraph-Level Chunking\n\n**How it works**: Each paragraph becomes a separate chunk.\n\n**Best for**:\n- Well-structured documents\n- Maintaining logical flow\n- When paragraphs represent distinct ideas\n\n**Example**: Each paragraph treated as independent unit",
        "Hierarchical Chunking": "## Hierarchical Chunking\n\n**How it works**: Multi-level chunking: sections → paragraphs → sentences.\n\n**Best for**:\n- Complex, structured documents\n- Research papers and academic texts\n- When document structure matters\n\n**Example**: Research paper split by sections, then paragraphs, then sentences",
        "Recursive Text Splitter": "## Recursive Text Splitter\n\n**How it works**: Recursively splits text by different separators until chunks are small enough.\n\n**Best for**:\n- Mixed-content documents\n- Automatic optimal chunking\n- When document structure varies\n\n**Example**: Split by sections, then paragraphs, then sentences as needed",
        "Character Text Splitter": "## Character Text Splitter\n\n**How it works**: Splits text by character count, regardless of word boundaries.\n\n**Best for**:\n- Strict character limits\n- Token-limited models\n- When exact character counts matter\n\n**Example**: 50-character chunks with 10-character overlap",
        "Token Text Splitter": "## Token Text Splitter\n\n**How it works**: Splits text based on token count using spaCy tokenization.\n\n**Best for**:\n- LLM input preparation\n- Transformer models\n- When token count matters more than word count\n\n**Example**: 100-token chunks with 20-token overlap"
    }

    def analyze_text(text, chunking_method, chunk_size, overlap, sentences_per_chunk):
        if not text.strip():
            return "Please enter some text to analyze.", {}, "", "No text to analyze"

        if chunking_method == "Fixed-Size Chunking":
            result = chunker.fixed_size_chunking(text, chunk_size, overlap)
        elif chunking_method == "Sentence-Based Chunking":
            result = chunker.sentence_based_chunking(text, sentences_per_chunk)
        elif chunking_method == "Sliding Window Chunking":
            result = chunker.sliding_window_chunking(text, chunk_size, overlap)
        elif chunking_method == "Paragraph-Level Chunking":
            result = chunker.paragraph_level_chunking(text)
        elif chunking_method == "Hierarchical Chunking":
            result = chunker.hierarchical_chunking(text, chunk_size)
        elif chunking_method == "Recursive Text Splitter":
            result = chunker.recursive_text_splitter(text, chunk_size, overlap)
        elif chunking_method == "Character Text Splitter":
            result = chunker.character_text_splitter(text, chunk_size, overlap)
        elif chunking_method == "Token Text Splitter":
            result = chunker.token_text_splitter(text, chunk_size, overlap)
        else:
            return "Method not implemented.", {}, "", ""

        stats = result.metadata or {}

        # 1. Chunks Display Layout
        chunks_display = ""
        for i, chunk in enumerate(result.chunks):
            words = len(chunk.split())
            chars = len(chunk)
            chunks_display += f"### Chunk {i+1} (Words: {words}, Chars: {chars})\n"
            chunks_display += f"<details><summary>Expand Contents</summary>\n\n{chunk}\n\n</details>\n\n---\n\n"

        # 2. Statistics Table Layout
        stats_table = f"## 📊 {result.method} Execution Statistics\n\n"
        stats_table += "| Metric Parameter | Aggregated Value |\n"
        stats_table += "| :--- | :--- |\n"
        stats_table += f"| **Total Chunks** | {len(result.chunks)} |\n"
        stats_table += f"| **Total Document Words** | {len(text.split())} |\n"
        stats_table += f"| **Total Document Characters** | {len(text)} |\n"

        for key, value in stats.items():
            formatted_key = key.replace('_', ' ').title()
            stats_table += f"| **{formatted_key}** | {value} |\n"

        # 3. Clean Fixed Token Layout (Uses native python html.escape safely)
        tokenization_display = f"## 🔤 Tokenization Snapshots ({result.method})\n\n"
        if result.tokens_examples:
            for i, tokens in enumerate(result.tokens_examples):
                tokenization_display += f"### Chunk {i+1} ({len(tokens)} tokens tracked)\n"

                flat_tokens = []
                for t in tokens:
                    if isinstance(t, list):
                        flat_tokens.extend([str(item) for item in t])
                    else:
                        flat_tokens.append(str(t))

                token_spans = " ".join([f"<code>{html.escape(t)}</code>" for t in flat_tokens])
                tokenization_display += f"<div style='background:#111; padding:12px; border-radius:6px; line-height:1.8; word-break:break-word;'>{token_spans}</div>\n\n---\n"
        else:
            tokenization_display += "\n*No operational token slices generated for this sequence layout.*"

        return chunks_display, stats, stats_table, tokenization_display

    def update_ui_elements(method):
        show_size_and_overlap = ["Fixed-Size Chunking", "Sliding Window Chunking", "Recursive Text Splitter", "Character Text Splitter", "Token Text Splitter"]
        show_just_size = ["Hierarchical Chunking"]
        show_sentences = ["Sentence-Based Chunking"]

        desc_update = gr.update(value=descriptions.get(method, ""))
        size_update = gr.update(visible=method in (show_size_and_overlap + show_just_size))
        overlap_update = gr.update(visible=method in show_size_and_overlap)
        sentences_update = gr.update(visible=method in show_sentences)

        return desc_update, size_update, overlap_update, sentences_update

    with gr.Blocks(theme=gr.themes.Soft()) as demo:
        with gr.Row():
            with gr.Column(scale=2):
                text_input = gr.Textbox(
                    label="Input Document",
                    value=sample_texts["Research Paper (AI in Healthcare)"],
                    lines=11,
                    placeholder="Provide text to split..."
                )

                sample_dropdown = gr.Dropdown(
                    choices=list(sample_texts.keys()),
                    value="Research Paper (AI in Healthcare)",
                    label="Preload Standard Datasets"
                )

                method_dropdown = gr.Dropdown(
                    choices=[
                        "Fixed-Size Chunking",
                        "Sentence-Based Chunking",
                        "Sliding Window Chunking",
                        "Paragraph-Level Chunking",
                        "Hierarchical Chunking",
                        "Recursive Text Splitter",
                        "Character Text Splitter",
                        "Token Text Splitter"
                    ],
                    value="Fixed-Size Chunking",
                    label="Splitting Engine Variant"
                )

                with gr.Group():
                    chunk_size = gr.Slider(minimum=10, maximum=500, value=100, step=10, label="Window Length Constraint")
                    overlap = gr.Slider(minimum=0, maximum=100, value=20, step=5, label="Boundary Intersect Overlap")
                    sentences_per_chunk = gr.Slider(minimum=1, maximum=50, value=10, step=1, label="Sentences Per Vector Group", visible=False)

                analyze_btn = gr.Button("🔍 Execute Text Segmentation", variant="primary")

            with gr.Column(scale=3):
                with gr.Tabs():
                    with gr.TabItem("📄 Chunks Output"):
                        chunks_output = gr.Markdown()

                    with gr.TabItem("🔤 Tokenization"):
                        tokenization_output = gr.Markdown()

                    with gr.TabItem("📊 Statistics"):
                        stats_output = gr.Markdown()

                    with gr.TabItem("📈 Visual Summary"):
                        json_output = gr.JSON()

                method_desc = gr.Markdown(descriptions["Fixed-Size Chunking"])

        method_dropdown.change(
            fn=update_ui_elements,
            inputs=method_dropdown,
            outputs=[method_desc, chunk_size, overlap, sentences_per_chunk]
        )

        sample_dropdown.change(
            fn=lambda x: sample_texts[x],
            inputs=sample_dropdown,
            outputs=text_input
        )

        analyze_btn.click(
            fn=analyze_text,
            inputs=[text_input, method_dropdown, chunk_size, overlap, sentences_per_chunk],
            outputs=[chunks_output, json_output, stats_output, tokenization_output]
        )

    return demo

if __name__ == "__main__":
    demo = create_demo()
    demo.launch(server_name="127.0.0.1", inbrowser=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cc1d7d0305f90b207b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
